In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [2]:
LATENT_DIM = 100
BATCH_SIZE = 128
NUM_EPOCHS = 50
LR = 0.0002

In [3]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

loader = DataLoader(
    datasets.MNIST(root='./data', train=True, download=True, transform=transform),
    batch_size=BATCH_SIZE, shuffle=True
)

100%|██████████| 9.91M/9.91M [00:00<00:00, 38.1MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.13MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 10.3MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 16.1MB/s]


In [4]:
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(LATENT_DIM, 256 * 7 * 7)
        self.net = nn.Sequential(
            nn.BatchNorm2d(256),
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            nn.ConvTranspose2d(64, 1, 4, 1, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, z):
        x = self.fc(z).view(-1, 256, 7, 7)
        x = self.net(x)
        return nn.functional.interpolate(x, size=(28, 28))

In [5]:
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 64, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, True),
            nn.Conv2d(64, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, True),
            nn.Conv2d(128, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, True),
        )
        self.fc = nn.Sequential(nn.Linear(256 * 3 * 3, 1), nn.Sigmoid())

    def forward(self, x):
        return self.fc(self.net(x).view(x.size(0), -1))

In [ ]:
G = Generator().to(device)
D = Discriminator().to(device)

criterion = nn.BCELoss()
opt_G = optim.Adam(G.parameters(), lr=LR, betas=(0.5, 0.999))
opt_D = optim.Adam(D.parameters(), lr=LR, betas=(0.5, 0.999))

g_losses, d_losses = [], []

for epoch in range(1, NUM_EPOCHS + 1):
    g_loss_epoch, d_loss_epoch = 0, 0
    for real, _ in loader:
        real = real.to(device)
        b = real.size(0)

        opt_D.zero_grad()
        d_real = criterion(D(real), torch.ones(b, 1).to(device))
        z = torch.randn(b, LATENT_DIM).to(device)
        d_fake = criterion(D(G(z).detach()), torch.zeros(b, 1).to(device))
        d_loss = (d_real + d_fake) / 2
        d_loss.backward()
        opt_D.step()

        opt_G.zero_grad()
        z = torch.randn(b, LATENT_DIM).to(device)
        g_loss = criterion(D(G(z)), torch.ones(b, 1).to(device))
        g_loss.backward()
        opt_G.step()

        g_loss_epoch += g_loss.item()
        d_loss_epoch += d_loss.item()

    g_losses.append(g_loss_epoch / len(loader))
    d_losses.append(d_loss_epoch / len(loader))

    if epoch % 5 == 0:
        print(f'Epoch {epoch}/{NUM_EPOCHS}  D: {d_losses[-1]:.4f}  G: {g_losses[-1]:.4f}')

Epoch 5/50  D: 0.1923  G: 2.6901
Epoch 10/50  D: 0.1689  G: 2.9909
Epoch 15/50  D: 0.1345  G: 3.2815
Epoch 20/50  D: 0.1532  G: 3.3152
Epoch 25/50  D: 0.1478  G: 3.5020
Epoch 30/50  D: 0.1091  G: 3.8624


In [ ]:
G.eval()
with torch.no_grad():
    samples = G(torch.randn(16, LATENT_DIM).to(device)).cpu()

grid = make_grid(samples, nrow=4, normalize=True).permute(1, 2, 0).numpy()
plt.figure(figsize=(6, 6))
plt.imshow(grid, cmap='gray')
plt.axis('off')
plt.show()